# EIA-176 Company Characteristics: Data Exploration

This notebook explores the source data for `core_eia176__yearly_company_characteristics`
(issue #4697), which captures Form EIA-176 Part 3 — Lines A–F — describing company
type, ownership, and operational characteristics.

**Source assets:**
- `_core_eia176__yearly_company_data` — intermediate wide table derived from the NGQS
  custom report (`raw_eia176__numeric_data`). Already contains boolean `is_*` columns
  for operation/ownership type.
- `raw_eia176__operation_types_and_sector_items` — separate extract from the
  `operation_types_and_sector_items` CSV page. May contain overlapping or complementary
  columns for different year ranges.

**Goals:**
1. Identify all Lines A–F columns available in each source.
2. Understand year coverage and null rates per column.
3. Check PK uniqueness `(report_year, operator_id_eia)`.
4. Characterize raw values in boolean/categorical columns (X, 0/1, yes/no, etc.).
5. Verify state normalization path via `core_pudl__codes_subdivisions`.
6. Decide merge strategy and document any quirks for the transform.

In [ ]:
import os

assert os.environ.get("DAGSTER_HOME"), (
    "DAGSTER_HOME is not set. Kill the jupyter server, set the env var, and relaunch."
)

In [ ]:
import pandas as pd
from dagster import AssetKey

from pudl.etl import defs

## 1. Load source assets

In [ ]:
company_data = defs.load_asset_value(AssetKey("_core_eia176__yearly_company_data"))
print(f"_core_eia176__yearly_company_data shape: {company_data.shape}")
print(f"Year range: {company_data['report_year'].min()} – {company_data['report_year'].max()}")
company_data.head(3)

In [ ]:
op_types = defs.load_asset_value(AssetKey("raw_eia176__operation_types_and_sector_items"))
print(f"raw_eia176__operation_types_and_sector_items shape: {op_types.shape}")
print(f"Year range: {op_types['report_year'].min()} – {op_types['report_year'].max()}")
op_types.head(3)

In [ ]:
subdivisions = defs.load_asset_value(AssetKey("core_pudl__codes_subdivisions"))
print(f"core_pudl__codes_subdivisions shape: {subdivisions.shape}")

## 2. Identify Lines A–F columns in each source

EIA-176 Part 3 Lines A–F from the survey form:
- **A** — Type of natural gas (conventional/unconventional) — *requires archiver work, skip for now*
- **B** — Type of business operations (distribution, pipeline, producer, etc.) → `is_*` boolean columns
- **C** — Ownership type (investor-owned, cooperative, municipal, private, other) → more `is_*` columns
- **D** — Alternative fuel vehicle fleet → `alternative_fuel_fleet_*` column
- **E/F** — Investigate availability below

Key boolean/categorical columns expected in `_core_eia176__yearly_company_data`:

In [ ]:
# Inspect all columns of _core_eia176__yearly_company_data
print("All columns:")
for col in sorted(company_data.columns):
    print(f"  {col}")

In [ ]:
# Filter to likely Lines A-F columns (is_* prefix + alternative_fuel + ownership)
lines_af_cols = [c for c in company_data.columns if c.startswith("is_") or "alternative_fuel" in c or "other_ownership" in c]
print(f"Lines A-F candidate columns in _core_eia176__yearly_company_data ({len(lines_af_cols)}):")
for c in lines_af_cols:
    print(f"  {c}")

In [ ]:
# Inspect all columns of raw_eia176__operation_types_and_sector_items
print("raw_eia176__operation_types_and_sector_items columns:")
for col in sorted(op_types.columns):
    print(f"  {col}")

In [ ]:
# Columns in op_types not in company_data (and vice versa)
cd_cols = set(company_data.columns)
ot_cols = set(op_types.columns)
pk_cols = {"report_year", "operator_id_eia", "operating_state", "operator_name"}

only_in_op_types = ot_cols - cd_cols - pk_cols
only_in_company = cd_cols - ot_cols - pk_cols
shared = (cd_cols & ot_cols) - pk_cols

print(f"Only in op_types ({len(only_in_op_types)}): {sorted(only_in_op_types)}")
print(f"Shared non-PK cols ({len(shared)}): {sorted(shared)}")
# Note: only_in_company is large (all the numeric columns) — not printing

## 3. Primary key analysis

In [ ]:
pk = ["report_year", "operator_id_eia"]

# PK uniqueness in company_data
dupes_cd = company_data[company_data.duplicated(subset=pk, keep=False)]
print(f"_core_eia176__yearly_company_data duplicate PK rows: {len(dupes_cd)}")
if not dupes_cd.empty:
    print(dupes_cd[pk + lines_af_cols[:5]].head(10))

In [ ]:
# PK uniqueness in op_types
pk_ot = ["report_year", "operator_id_eia"]
dupes_ot = op_types[op_types.duplicated(subset=pk_ot, keep=False)]
print(f"raw_eia176__operation_types_and_sector_items duplicate PK rows: {len(dupes_ot)}")
if not dupes_ot.empty:
    print(dupes_ot.head(10))

In [ ]:
# Year coverage: how many operators per year in each source
cd_yearly = company_data.groupby("report_year").size().rename("company_data")
ot_yearly = op_types.groupby("report_year").size().rename("op_types")
pd.concat([cd_yearly, ot_yearly], axis=1)

## 4. Null rates and value distributions

In [ ]:
# Null rates for Lines A-F columns in company_data
null_rates = company_data[lines_af_cols].isna().mean().sort_values(ascending=False)
print("Null rates for Lines A-F columns in _core_eia176__yearly_company_data:")
print(null_rates.to_string())

In [ ]:
# Raw value distributions for all Lines A-F columns — looking for X, 0/1, yes/no, etc.
for col in lines_af_cols:
    vc = company_data[col].value_counts(dropna=False)
    print(f"\n--- {col} ---")
    print(vc.head(10).to_string())

In [ ]:
# Same for op_types columns (excluding shared PK cols)
ot_value_cols = [c for c in op_types.columns if c not in pk_cols]
for col in ot_value_cols:
    vc = op_types[col].value_counts(dropna=False)
    print(f"\n--- {col} (op_types) ---")
    print(vc.head(10).to_string())

## 5. Year-by-year availability of boolean columns

Some boolean columns were added in later years. Check which columns are entirely null in
earlier years.

In [ ]:
# For each Lines A-F column, show first and last year with a non-null value
for col in lines_af_cols:
    non_null = company_data.dropna(subset=[col])
    if non_null.empty:
        print(f"{col}: entirely null")
    else:
        print(f"{col}: {int(non_null['report_year'].min())}–{int(non_null['report_year'].max())} ({len(non_null)} rows)")

## 6. State normalization

The `operating_state` column contains full names (e.g., "texas"). We normalize to
two-letter subdivision codes via `core_pudl__codes_subdivisions`, reusing the existing
`_normalize_operating_states` helper.

In [ ]:
# What does operating_state look like?
print("Unique operating_state values in company_data (sample):")
print(sorted(company_data["operating_state"].dropna().unique())[:30])

In [ ]:
# Build normalization map and check coverage
from pudl.transform.eia176 import DROP_OPERATING_STATES, _normalize_operating_states

codes = (
    subdivisions.assign(key=lambda d: d["subdivision_name"].str.strip().str.casefold())
    .drop_duplicates("key")
    .set_index("key")["subdivision_code"]
)

norm = company_data["operating_state"].astype(str).str.strip().str.casefold()
not_covered = norm[~norm.isin(codes.index) & ~norm.isin(DROP_OPERATING_STATES)].unique()
print(f"operating_state values not in subdivision codes or DROP list: {not_covered}")

In [ ]:
# How many rows would be dropped by null operating_state after normalization?
df_norm = _normalize_operating_states(subdivisions, company_data[["report_year", "operator_id_eia", "operating_state"]])
null_state_rows = df_norm["operating_state"].isna().sum()
total_rows = len(df_norm)
print(f"Rows with null operating_state after normalization: {null_state_rows} / {total_rows} ({null_state_rows/total_rows:.1%})")

## 7. Merge strategy: company_data vs. op_types

Decide whether to pull boolean columns from `_core_eia176__yearly_company_data` directly
(simpler, already cleaned) or join with `raw_eia176__operation_types_and_sector_items`
(potentially more complete for certain years or additional columns).

In [ ]:
# Check: for shared columns, do company_data and op_types agree?
# Merge on (report_year, operator_id_eia) and compare shared non-PK columns
shared_cols = list(shared)  # defined above

if shared_cols:
    merge_key = ["report_year", "operator_id_eia"]
    merged = company_data[merge_key + shared_cols].merge(
        op_types[merge_key + shared_cols],
        on=merge_key,
        suffixes=("_cd", "_ot"),
        how="inner",
    )
    for col in shared_cols:
        cd_col, ot_col = f"{col}_cd", f"{col}_ot"
        if cd_col in merged.columns:
            mismatches = (merged[cd_col] != merged[ot_col]).sum()
            print(f"{col}: {mismatches} mismatches between sources")
else:
    print("No shared non-PK columns between sources — merge would only add new columns.")

In [ ]:
# How many op_types rows have no matching row in company_data (and vice versa)?
merge_key = ["report_year", "operator_id_eia"]
cd_keys = company_data[merge_key].drop_duplicates()
ot_keys = op_types[merge_key].drop_duplicates()

only_cd = cd_keys.merge(ot_keys, how="left", indicator=True).query('_merge == "left_only"')
only_ot = ot_keys.merge(cd_keys, how="left", indicator=True).query('_merge == "left_only"')

print(f"Rows in company_data with no op_types match: {len(only_cd)}")
print(f"Rows in op_types with no company_data match: {len(only_ot)}")

if not only_ot.empty:
    print("\nop_types-only rows by year:")
    print(only_ot["report_year"].value_counts().sort_index())

## 8. Boolean conversion candidates

Identify which columns need `X`/yes-no → `True`/`False` conversion vs. which are already
0/1 floats (from the numeric NGQS custom report).

In [ ]:
for col in lines_af_cols:
    unique_vals = set(company_data[col].dropna().astype(str).str.strip().str.lower().unique())
    dtype = company_data[col].dtype
    print(f"{col} [{dtype}]: {unique_vals}")

## 9. Proposed final column set

Select the columns to include in `core_eia176__yearly_company_characteristics`, excluding
Line A (pending archiver work) and any columns found to be entirely null or unavailable.

In [ ]:
PRIMARY_KEY = ["report_year", "operator_id_eia"]

# Proposed characteristics columns — adjust after exploring above
CHARS_COLS = [
    # Line B: type of business operations
    "is_distribution_company_cooperative",
    "is_distribution_company_investor_owned",
    "is_distribution_company_municipally_owned",
    "is_distribution_company_privately_owned",
    "is_gatherer",
    "is_interstate_pipeline",
    "is_intrastate_pipeline",
    "is_liquid_natural_gas_marine_terminal",
    "is_liquid_natural_gas_peak_facility_operator",
    "is_producer",
    "is_storage_operator",
    "is_synthetic_natural_gas_plant_operator",
    # Line C: ownership
    "is_other_ownership",
    "other_ownership_description",
    # Line D: alternative fuel fleet (present in some years)
    # (discover exact column name from exploration above)
    # CNG/LNG fueling stations (added ~2016)
    "is_public_compressed_natural_gas_fueling_station",
    "is_public_liquid_natural_gas_fueling_station",
]

# Filter to columns actually present
available_chars = [c for c in CHARS_COLS if c in company_data.columns]
missing_chars = [c for c in CHARS_COLS if c not in company_data.columns]
print(f"Available: {available_chars}")
print(f"Missing (check op_types): {missing_chars}")

In [ ]:
# Build a draft characteristics table from company_data alone
df_chars = company_data[PRIMARY_KEY + ["operating_state"] + available_chars].copy()
print(f"Draft shape: {df_chars.shape}")
print(f"Duplicate PK rows: {df_chars.duplicated(subset=PRIMARY_KEY).sum()}")
df_chars.head()

## 10. Smoke test: apply transforms

Once `core_eia176__yearly_company_characteristics` is implemented in
`src/pudl/transform/eia176.py`, load and validate it here.

In [ ]:
# Uncomment after the transform asset is implemented:
# result = defs.load_asset_value(AssetKey("core_eia176__yearly_company_characteristics"))
# print(f"Shape: {result.shape}")
# print(result.dtypes)
# print(f"Duplicate PK: {result.duplicated(subset=PRIMARY_KEY).sum()}")
# result.head()

In [ ]:
# Row counts per year (for dbt etl_full_row_counts.csv)
# Uncomment after transform is wired:
# result.groupby("report_year").size().rename("n_rows")

## 11. Non-standard null check in op_types boolean columns

Raw CSVs sometimes encode false as `"-"`, `"0"`, `"n"`, empty string, or whitespace.
Check every `is_*` column and `other_ownership_description` for unexpected values.

In [ ]:
IS_COLS = [c for c in op_types.columns if c.startswith("is_")]
TEXT_COLS = ["other_ownership_description"]

# For boolean columns: anything that is not "X" or NaN is suspicious
print("=== Unexpected values in is_* columns ===")
for col in IS_COLS:
    unexpected = op_types[col].dropna()
    unexpected = unexpected[unexpected != "X"]
    if not unexpected.empty:
        print(f"{col}: {unexpected.value_counts().to_dict()}")
    else:
        print(f"{col}: OK (only X / NaN)")

print("\n=== other_ownership_description sample values ===")
print(op_types["other_ownership_description"].value_counts(dropna=False).head(20))

## 12. Clarify `is_other_ownership_2`

The form has one 'other' ownership checkbox. Investigate whether `is_other_ownership_2`
is a second checkbox or a data artifact.

In [ ]:
# How often are both is_other_ownership and is_other_ownership_2 marked?
both = op_types[(op_types["is_other_ownership"] == "X") & (op_types["is_other_ownership_2"] == "X")]
only_2 = op_types[(op_types["is_other_ownership"].isna()) & (op_types["is_other_ownership_2"] == "X")]
only_1 = op_types[(op_types["is_other_ownership"] == "X") & (op_types["is_other_ownership_2"].isna())]

print(f"Both is_other_ownership AND is_other_ownership_2 marked X: {len(both)}")
print(f"Only is_other_ownership_2 marked X (is_other_ownership NaN): {len(only_2)}")
print(f"Only is_other_ownership marked X (is_other_ownership_2 NaN): {len(only_1)}")

print("\nYear distribution of 'only_2' rows:")
print(only_2["report_year"].value_counts().sort_index())

print("\nSample 'only_2' rows (operator + description):")
print(only_2[["report_year", "operator_id_eia", "operator_name", "is_other_ownership", "is_other_ownership_2", "other_ownership_description"]].head(10).to_string())

In [ ]:
# Investigate the 75 rows where other_ownership_description is 1.0 (float)
float_desc = op_types[op_types["other_ownership_description"] == 1.0]
print(f"Rows with other_ownership_description == 1.0: {len(float_desc)}")
print(f"Year range: {float_desc['report_year'].min():.0f}–{float_desc['report_year'].max():.0f}")
print(f"\nYear distribution:")
print(float_desc["report_year"].value_counts().sort_index())
print(f"\nWhich is_* columns are marked for these rows:")
IS_COLS = [c for c in op_types.columns if c.startswith('is_')]
print(float_desc[IS_COLS].apply(lambda c: (c == 'X').sum()).sort_values(ascending=False))
print(f"\nSample rows:")
print(float_desc[["report_year", "operator_id_eia", "operator_name", "operating_state",
                  "is_other_ownership", "is_other_ownership_2", "other_ownership_description"]].head(15).to_string())

## 13. Build merged draft table

Left-join `op_types` (primary source, all `is_*` columns) with the two columns
we need from `_core_eia176__yearly_company_data`: `operating_state` and
`alternative_fuel_fleet_1_yes_0_no`.

In [ ]:
PK = ["report_year", "operator_id_eia"]
IS_COLS = [c for c in op_types.columns if c.startswith("is_")]
# operating_state in op_types is already 2-letter postal codes — use it directly
# Only pull alternative_fuel_fleet_1_yes_0_no from company_data
OP_KEEP = PK + ["operator_name", "operating_state"] + IS_COLS + ["other_ownership_description"]
CD_KEEP = PK + ["alternative_fuel_fleet_1_yes_0_no"]

df = op_types[OP_KEEP].merge(
    company_data[CD_KEEP],
    on=PK,
    how="left",
    validate="1:1",
)

print(f"Merged shape: {df.shape}")
print(f"Expected: ({len(op_types)}, {len(OP_KEEP) + len(CD_KEEP) - len(PK)})")
print(f"Duplicate PK rows: {df.duplicated(subset=PK).sum()}")
print(f"Rows where operating_state is null: {df['operating_state'].isna().sum()}")
print(f"\nSample operating_state values (op_types sourced):")
print(df['operating_state'].value_counts().head(10))
df.head(3)

## 14. Apply transforms

1. `"X" → True`, `NaN → False` for all `is_*` columns  
2. Rename and convert `alternative_fuel_fleet_1_yes_0_no → has_alternative_fuel_fleet`  
3. Normalize `operating_state` to two-letter codes  
4. Drop rows where `operating_state` is null after normalization

In [ ]:
df_t = df.copy()

# 1. Convert X-encoded boolean columns
for col in IS_COLS:
    df_t[col] = df_t[col].eq("X")

# 2. Rename and convert alternative fuel fleet
df_t = df_t.rename(columns={"alternative_fuel_fleet_1_yes_0_no": "has_alternative_fuel_fleet"})
df_t["has_alternative_fuel_fleet"] = df_t["has_alternative_fuel_fleet"].eq(1.0)

# 3. operating_state from op_types is already 2-letter codes — no normalization needed
# Drop rows with null operating_state (operators with no state recorded)
n_before = len(df_t)
df_t = df_t.dropna(subset=["operating_state"])
print(f"Dropped {n_before - len(df_t)} rows with null operating_state")
print(f"Final shape: {df_t.shape}")

# Confirm dtypes
bool_cols = IS_COLS + ["has_alternative_fuel_fleet"]
print("\nBoolean column dtypes:")
print(df_t[bool_cols].dtypes.value_counts())

## 15. Final validation

In [ ]:
# PK uniqueness
dupes = df_t.duplicated(subset=PK).sum()
print(f"Duplicate PK rows: {dupes}")
assert dupes == 0, "Unexpected duplicate PKs"

# Boolean columns contain only True/False
for col in bool_cols:
    bad = df_t[col].isna().sum()
    if bad:
        print(f"WARNING: {col} has {bad} NaN values")
assert df_t[bool_cols].isna().sum().sum() == 0, "NaN found in boolean columns"
print("All boolean columns contain only True/False — OK")

# operating_state is all two-letter codes
bad_states = df_t["operating_state"].str.len().ne(2).sum()
print(f"operating_state values with length != 2: {bad_states}")

# True rate per boolean column (to spot columns that are always False)
print("\nTrue rate per boolean column:")
true_rates = df_t[bool_cols].mean().sort_values(ascending=False)
print(true_rates.to_string())

In [ ]:
# Row counts per year — will be needed for dbt etl_full_row_counts.csv
print("Row counts per year (for dbt seed):")
print(df_t.groupby("report_year").size().rename("n_rows").to_string())